# CrewAI Patterns — Agents, Tasks, Crews, and Process Types
## Processes, Tasks, Crews — CrewAI Architecture Patterns
⏱ ~15 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/23-crewai-notebook/crewai_workbook.ipynb)

**CrewAI** is a multi-agent framework built on the **role-based organization** metaphor. Where LangGraph asks you to define a state machine (nodes + edges), CrewAI asks you to describe a *team*: who are the agents, what are their roles, what tasks need doing, and how should the crew work together? By the end of this session you will understand *why* the crew abstraction exists, *how* every component fits together, and *how* to build production-grade multi-agent pipelines.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Agents, Tasks, Crews, and Process types |
| 2 | **Agents** — Role, goal, backstory, and LLM configuration |
| 3 | **Tasks and Sequential Process** — Ordered handoffs between agents |
| 4 | **Hierarchical Process** — Manager-directed dynamic routing |
| 5 | **Custom Tools** — Extending agents beyond the LLM |
| 6 | **Crew Output and Inspection** — Reading results programmatically |
| 7 | **Debugging and Evaluation** — Know when CrewAI fails |
| ★ | **Exercises, Answer Key, and What's Next** |

---

### Prerequisites
- Python 3.10+ (a `.venv` with the requirements already installed)
- An `OPENAI_API_KEY` set in `.env` (or Colab Secrets)

### Key References
> Hong, S., et al. (2023). *MetaGPT: Meta Programming for a Multi-Agent Collaborative Framework.* https://arxiv.org/abs/2308.00352  
> Wu, Q., et al. (2023). *AutoGen: Enabling Next-Gen LLM Applications via Multi-Agent Conversation.* https://arxiv.org/abs/2308.08155  
> CrewAI documentation: https://docs.crewai.com  
> CrewAI GitHub: https://github.com/joaomdmoura/crewAI

In [ ]:
# Install dependencies — runs only on Google Colab.
# Local users: your .venv already has everything from requirements.txt.
import sys


def _in_colab():
    try:
        import google.colab

        return True
    except ImportError:
        return False


if _in_colab():
    import subprocess

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "crewai>=0.76.0",
            "langchain-openai==0.3.33",
            "python-dotenv==1.1.1",
            "ddgs",
            # "duckduckgo-search",  # legacy package — langchain-community 0.3.31+ requires ddgs instead
        ],
        check=True,
    )
    print("Colab install complete.")
else:
    print("Local environment detected — skipping install.")

In [ ]:
import os

# ── API key ───────────────────────────────────────────────────────────────────
# Colab: set in Secrets panel (left sidebar key icon)
# Local: create a .env file containing  OPENAI_API_KEY=sk-...
try:
    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv

    load_dotenv()

# ── Sanity check ──────────────────────────────────────────────────────────────
key = os.environ.get("OPENAI_API_KEY", "")
key_ok = bool(key) and key.startswith("sk-")
print(f"OPENAI_API_KEY present and valid: {key_ok}")
if not key_ok:
    print()
    print("  ACTION REQUIRED — add your key before running any LLM cells.")
    print("  Local: echo 'OPENAI_API_KEY=sk-...' >> .env")
    print("  Colab: Secrets panel → Add secret 'OPENAI_API_KEY'")

---

## Part 1 — What Is CrewAI and Why Does It Exist?

---

### The Problem with Single-Agent Systems

A single LLM agent can only maintain one perspective. When a task requires multiple specialists — research, analysis, writing, fact-checking — a single agent tries to do everything and does none of it well. Role confusion produces mediocre output.

**CrewAI's answer**: model the problem as a team, not a state machine.

---

### Three Multi-Agent Framework Models Compared

| Framework | Mental model | Control flow | Best for |
|-----------|-------------|-------------|----------|
| **LangGraph** | State machine | Graph edges (you wire it) | Precise, auditable pipelines |
| **CrewAI** | Organization chart | Process type (crew handles it) | Role-based team collaboration |
| **AutoGen** | Chat room | Conversation turns | Free-form agent dialogue |

---

### CrewAI Core Vocabulary

| Term | Definition |
|------|------------|
| **Agent** | An LLM instance with a role, goal, backstory, and optional tools |
| **Task** | A unit of work: description, expected output, and assigned agent |
| **Crew** | The container that holds agents + tasks and orchestrates execution |
| **Process** | The orchestration strategy: `sequential` or `hierarchical` |
| **Tool** | A callable function agents can invoke to interact with the world |
| **Backstory** | The LLM persona — shapes reasoning style more than role alone |
| **Context** | Task-to-task output passing — the handoff mechanism |
| **Manager agent** | Auto-created in hierarchical mode to route tasks dynamically |

### CrewAI Architecture

```
SEQUENTIAL PROCESS (ordered, deterministic)
─────────────────────────────────────────────────────

  Crew.kickoff()
       │
       ▼
  ┌──────────────┐
  │   Task 1     │  agent: researcher
  │  (research)  │  output: bullet list
  └──────┬───────┘
         │  context passed automatically
         ▼
  ┌──────────────┐
  │   Task 2     │  agent: writer
  │   (write)    │  context: [task_1]
  └──────┬───────┘  output: article draft
         │
         ▼
  CrewOutput (result.raw / result.tasks_output)


HIERARCHICAL PROCESS (manager-directed, dynamic)
─────────────────────────────────────────────────────

  Crew.kickoff()
       │
       ▼
  ┌──────────────┐
  │ Manager Agent│  auto-created by CrewAI
  │  (LLM plays  │  reads all tasks + agents
  │  the boss)   │  decides routing + order
  └──────┬───────┘
     ┌───┴───────┐
     ▼           ▼
  researcher   writer
  (assigned    (assigned
   by manager)  by manager)
```

> **LangGraph comparison:** `sequential` maps to a linear graph with fixed edges. `hierarchical` maps to the supervisor/worker pattern from example 19 — but CrewAI auto-creates the supervisor and writes the routing logic, while LangGraph requires you to code it explicitly.

---

## Part 2 — Agents: The Building Blocks

---

An **Agent** in CrewAI is defined by four required fields plus optional configuration:

| Field | What it does | Why it matters |
|-------|-------------|----------------|
| `role` | The job title | Focuses the LLM on a domain; appears in task routing |
| `goal` | What the agent is trying to achieve | Sets the objective for every action it takes |
| `backstory` | Background and expertise | **Most underrated field** — shapes reasoning style and persona |
| `llm` | The language model to use | Lets you use different models per agent (cost vs. quality) |
| `tools` | List of callable tools | Extends the agent beyond pure LLM capabilities |
| `verbose` | Print reasoning steps | Useful for debugging; disable in production |
| `allow_delegation` | Can assign sub-tasks | Enables agent-to-agent delegation |
| `max_iter` | Max reasoning iterations | Guards against runaway loops |

### Backstory Matters More Than You Think

A generic backstory produces generic output. Compare:

```python
# Weak backstory — LLM has no anchoring
backstory="You are a helpful assistant."

# Strong backstory — LLM anchors to a specific persona
backstory=(
    "You are a senior technology analyst with 10 years of experience tracking "
    "AI and software trends. You synthesize complex technical topics into clear, "
    "accurate summaries backed by specific examples and data points."
)
```

The backstory is injected into every prompt the agent receives — think of it as a persistent system prompt.

In [ ]:
# ─── 2-A: Define agents with contrasting roles ────────────────────────────────
from crewai import LLM, Agent

llm = LLM(model="openai/gpt-4o-mini")

# A researcher with a specific domain and expertise
researcher = Agent(
    role="Technology Trend Researcher",
    goal="Find and summarize key developments in AI and software engineering",
    backstory=(
        "You are a senior technology analyst with 10 years of experience tracking "
        "AI and software trends. You synthesize complex technical topics into clear, "
        "accurate summaries backed by specific examples."
    ),
    llm=llm,
    verbose=False,
)

# A writer with a contrasting voice
writer = Agent(
    role="Technical Content Writer",
    goal="Write engaging, accurate technical content for developer audiences",
    backstory=(
        "You are a developer advocate who writes for engineers. You avoid jargon, "
        "prefer concrete examples over abstractions, and always include a practical "
        "takeaway in every piece you write."
    ),
    llm=llm,
    verbose=False,
)

print(f"Researcher: '{researcher.role}'")
print(f"Goal:        {researcher.goal}")
print()
print(f"Writer:      '{writer.role}'")
print(f"Goal:         {writer.goal}")

In [ ]:
# ─── 2-B: Inspect what CrewAI sends to the LLM ──────────────────────────────
# The system prompt CrewAI constructs from role + goal + backstory.
# This is what the LLM sees as its identity on every turn.

print("=== SYSTEM PROMPT PREVIEW (researcher) ===")
print()
print(f"Role:      {researcher.role}")
print(f"Goal:      {researcher.goal}")
print(f"Backstory: {researcher.backstory}")
print()
print("CrewAI combines these into a system prompt like:")
print("-" * 60)
print(
    f"You are {researcher.role}.\n"
    f"Your goal is: {researcher.goal}\n"
    f"Background: {researcher.backstory}"
)

---

## Part 3 — Tasks and the Sequential Process

---

A **Task** defines a unit of work. It has three required fields and one critical optional one:

| Field | Required | What it does |
|-------|----------|--------------|
| `description` | Yes | What should be done — the assignment prompt |
| `expected_output` | Yes | What the output should look like — acceptance criteria |
| `agent` | Yes | Which agent performs this task |
| `context` | No | List of other tasks whose outputs feed into this one |

### Task Context — How Handoffs Work

```python
write_task = Task(
    ...
    context=[research_task],   # writer receives researcher's output automatically
)
```

When `context=[research_task]` is set, CrewAI appends the research task's raw output to the writer's task prompt. The writer sees the actual bullet points the researcher produced — not a summary, the full text.

### Sequential Process Flow

```
Crew.kickoff()
  → Task 1 executes (researcher)
      → Task 1 output saved to tasks_output[0]
  → Task 2 executes (writer)
      → Task 1 output injected into Task 2 prompt via context=[task_1]
      → Task 2 output saved to tasks_output[1]
  → Final: result.raw == Task N's output (the last task)
```

In [ ]:
# ─── 3-A: Define tasks ────────────────────────────────────────────────────────
from crewai import Crew, Process, Task

TOPIC = "the rise of LLM-based coding agents in 2024-2025"

research_task = Task(
    description=(
        f"Research {TOPIC}. "
        "Identify 3-5 key trends, notable tools (e.g. GitHub Copilot, Cursor, Claude Code), "
        "and their impact on software engineering workflows."
    ),
    expected_output="A bullet-point list of 3-5 key trends with one concrete example each.",
    agent=researcher,
)

write_task = Task(
    description=(
        f"Using the research provided, write a concise 150-word blog introduction about {TOPIC}. "
        "Open with a hook, cover 2 key trends, end with a call to action for developers."
    ),
    expected_output="A 150-word blog introduction paragraph. No headers, just flowing prose.",
    agent=writer,
    context=[research_task],  # writer receives researcher's output as context
)

print("Tasks defined:")
print(f"  Task 1: '{research_task.description[:60]}...'")
print(f"  Task 2: '{write_task.description[:60]}...'")
print(f"  Task 2 has context: {len(write_task.context)} upstream task(s)")

In [ ]:
# ─── 3-B: Assemble and run the sequential crew ────────────────────────────────

sequential_crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, write_task],
    process=Process.sequential,
    verbose=False,
)

print(f"Crew assembled: {len(sequential_crew.agents)} agents, {len(sequential_crew.tasks)} tasks")
print(f"Process: {sequential_crew.process}")
print("Running crew...")
print()

result = sequential_crew.kickoff()

print("=== FINAL OUTPUT (write_task) ===")
print(result.raw)

In [ ]:
# ─── 3-C: Inspect per-task outputs in CrewOutput ─────────────────────────────
# result.tasks_output is a list — one TaskOutput per task in order.
# This is how you read intermediate outputs, not just the final one.

print("=== TASKS OUTPUT BREAKDOWN ===")
print()
for i, task_output in enumerate(result.tasks_output):
    print(f"--- Task {i + 1}: {task_output.description[:55]}...")
    print(f"    Agent:  {task_output.agent}")
    preview = task_output.raw[:250].replace("\n", " ")
    print(f"    Output: {preview}...")
    print()

print(f"Token usage: {result.token_usage}")

---

## Part 4 — The Hierarchical Process

---

In a **hierarchical process**, CrewAI creates a **manager agent** automatically. The manager reads the full task list and all available agents, then decides which agent gets which task and in what order — without you wiring the routing.

### Sequential vs Hierarchical — Side by Side

| Dimension | Sequential | Hierarchical |
|-----------|-----------|-------------|
| Routing | Fixed: task order in the list | Dynamic: manager decides |
| Manager | None | Auto-created by CrewAI |
| Predictability | High — same order every run | Lower — manager may reorder |
| Extra LLM calls | None | +1 manager call per delegation |
| Best for | Defined pipelines with clear handoffs | Open-ended tasks, unclear routing |

### When to Use Hierarchical

- Tasks have ambiguous ordering (manager figures it out)
- Some tasks may be skippable depending on prior output
- You want the framework to handle delegation rather than coding it yourself

### LangGraph vs CrewAI Hierarchical

```python
# LangGraph supervisor (example 19) — you write the router function:
def route(state):
    if state["next"] == "researcher":
        return "researcher"
    return "writer"

# CrewAI hierarchical — the manager LLM writes the router:
crew = Crew(
    process=Process.hierarchical,
    manager_llm=llm,   # <-- this LLM plays the manager role
)
```

The trade-off: CrewAI is less code but less auditable. LangGraph requires more code but every routing decision is visible and testable.

In [ ]:
# ─── 4-A: Run the same tasks under hierarchical process ──────────────────────
# Note: same agents and tasks as Part 3 — only process and manager_llm change.

hierarchical_crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, write_task],
    process=Process.hierarchical,
    manager_llm=llm,   # the LLM that plays the manager role
    verbose=False,
)

print("Running hierarchical crew...")
print()

result_h = hierarchical_crew.kickoff()

print("=== HIERARCHICAL OUTPUT ===")
print(result_h.raw)

In [ ]:
# ─── 4-B: Compare sequential vs hierarchical token usage ─────────────────────

print("SEQUENTIAL final output (first 200 chars):")
print(result.raw[:200])
print()
print("HIERARCHICAL final output (first 200 chars):")
print(result_h.raw[:200])
print()
print(f"Sequential   token usage: {result.token_usage}")
print(f"Hierarchical token usage: {result_h.token_usage}")
print()
print("Observation: hierarchical uses more tokens because the manager")
print("adds an LLM call to determine routing before delegating tasks.")

---

## Part 5 — Custom Tools

---

Agents become far more useful when they have **tools** — callable functions that let them interact with the world beyond the LLM's training data.

### Tool Design Principles

| Principle | What it means |
|-----------|---------------|
| **Single responsibility** | One tool does one thing — don't combine search + summarize |
| **String in, string out** | Agent passes strings; your tool converts types internally |
| **Informative errors** | Return error messages as strings, not exceptions — the agent reads them |
| **Docstring is the prompt** | The docstring is what the agent reads to decide when to call the tool |
| **LangChain tools work** | Any `@tool` from `langchain_community.tools` is plug-and-play |

### How the Agent Decides to Use a Tool

```
Agent receives task description
       │
       ▼
  Reasoning loop:
  "Can I answer this from my training data alone?"
  → Yes: generate answer directly
  → No:  scan available tools
              │
              ▼
           Read each tool's docstring
           Pick the most relevant tool
           Call tool(input_string)
           Incorporate result into answer
```

The docstring is the tool's contract with the agent — write it as a description of *when* to call it, *what* to pass, and *what* comes back.

In [ ]:
# ─── 5-A: Build a custom tool with the @tool decorator ───────────────────────
from crewai.tools import tool

# A local knowledge base lookup — simulates a private database the LLM cannot access
KNOWLEDGE_BASE = {
    "langgraph": (
        "LangGraph is a framework for building stateful multi-agent applications "
        "using graph-based workflows. It models agents as nodes and control flow as edges."
    ),
    "crewai": (
        "CrewAI is a role-based multi-agent framework. Agents have roles, goals, and backstories. "
        "Tasks are assigned to agents. Crews orchestrate via sequential or hierarchical processes."
    ),
    "autogen": (
        "AutoGen is a Microsoft framework for multi-agent conversation. Agents send messages "
        "to each other directly via ConversableAgent. No explicit graph or crew needed."
    ),
    "openai": (
        "OpenAI provides GPT-4o, GPT-4o-mini, o1, and o3 models via API. "
        "Function calling and Assistants API support tool use natively."
    ),
}


@tool("lookup_framework")
def lookup_framework(framework_name: str) -> str:
    """Look up a technical description of an AI agent framework or LLM provider by name.
    Use this when you need accurate facts about frameworks like LangGraph, CrewAI, AutoGen, or OpenAI.
    Input: the framework name as a lowercase string (e.g. 'langgraph', 'crewai').
    Returns the description or an error if the framework is not found.
    """
    key = framework_name.lower().strip()
    result = KNOWLEDGE_BASE.get(key)
    if result:
        return result
    available = ", ".join(KNOWLEDGE_BASE.keys())
    return f"No entry found for '{framework_name}'. Available: {available}"


# Quick test — call the tool directly before attaching to an agent
print("Direct tool test (crewai):")
print(lookup_framework.run("crewai"))
print()
print("Direct tool test (django — not in KB):")
print(lookup_framework.run("django"))

In [ ]:
# ─── 5-B: Attach tool to an agent and run ────────────────────────────────────

framework_expert = Agent(
    role="AI Framework Specialist",
    goal="Accurately compare and explain AI agent frameworks using reliable reference data",
    backstory=(
        "You are a principal engineer who has built production systems with LangGraph, CrewAI, "
        "and AutoGen. You always use your lookup_framework tool to verify facts before "
        "making claims — you never rely on memory alone."
    ),
    llm=llm,
    tools=[lookup_framework],
    verbose=False,
)

compare_task = Task(
    description=(
        "Use your lookup_framework tool to look up both 'langgraph' and 'crewai'. "
        "Then write a concise 80-word comparison of their core design philosophies, "
        "highlighting what each framework optimizes for and when a developer should "
        "choose one over the other."
    ),
    expected_output="An 80-word comparison paragraph citing specific design choices from each framework.",
    agent=framework_expert,
)

solo_crew = Crew(
    agents=[framework_expert],
    tasks=[compare_task],
    process=Process.sequential,
    verbose=False,
)

result_tool = solo_crew.kickoff()
print(result_tool.raw)

In [ ]:
# ─── 5-C: A second tool — deterministic calculator ────────────────────────────
# Demonstrates that tools don't have to call external APIs.
# Any deterministic Python function is a valid tool.
import re


@tool("calculate_cost")
def calculate_cost(expression: str) -> str:
    """Evaluate a simple arithmetic expression involving agent cost calculations.
    Use this when the task requires computing numbers (e.g. token cost, agent count * price).
    Input: a Python arithmetic expression as a string (e.g. '1000 * 0.002').
    Returns the numeric result as a string, or an error message if the expression is invalid.
    Only accepts: digits, spaces, +, -, *, /, (, ), and decimal points.
    """
    if not re.match(r"^[\d\s\+\-\*\/\.\(\)]+$", expression):
        return f"Error: invalid expression '{expression}' — only numbers and operators allowed."
    try:
        return str(eval(expression))  # safe: input validated above
    except Exception as e:
        return f"Error evaluating '{expression}': {e}"


# Test the calculator tool directly before attaching to an agent
print("Calculator tool tests:")
tests = [
    "1000 * 0.002",        # valid: token cost
    "(5 + 3) * 12",        # valid: grouped expression
    "import os",           # rejected: non-numeric
]
for expr in tests:
    print(f"  Input:  '{expr}'")
    print(f"  Output: {calculate_cost.run(expr)}")
    print()

---

## Part 6 — Crew Output and Inspection

---

Understanding `CrewOutput` is essential for building reliable pipelines. `Crew.kickoff()` returns a `CrewOutput` object:

| Attribute | Type | What it contains |
|-----------|------|------------------|
| `result.raw` | `str` | The final task's raw text output |
| `result.tasks_output` | `list[TaskOutput]` | One `TaskOutput` per task, in order |
| `result.token_usage` | `dict` | Token counts for the full run |
| `result.json_dict` | `dict or None` | Parsed JSON if output is structured |
| `result.pydantic` | `BaseModel or None` | Validated Pydantic model if configured |

Each `TaskOutput` has:
- `.raw` — the text the agent produced
- `.agent` — which agent ran it
- `.description` — the task description
- `.expected_output` — what was requested

### Structured Output with `output_pydantic`

```python
from pydantic import BaseModel

class ResearchSummary(BaseModel):
    trends: list[str]
    tools: list[str]

research_task = Task(
    ...
    output_pydantic=ResearchSummary,  # forces structured output
)
# result.pydantic is a validated ResearchSummary instance
```

### Dynamic Inputs at `kickoff()` Time

```python
crew.kickoff(inputs={"topic": "quantum computing", "audience": "executives"})
# All {topic} and {audience} placeholders in task descriptions get replaced.
```

In [ ]:
# ─── 6-A: Structured output with output_pydantic ─────────────────────────────
from typing import List

from pydantic import BaseModel


class FrameworkComparison(BaseModel):
    frameworks: List[str]
    summary: str
    recommendation: str


structured_task = Task(
    description=(
        "Use the lookup_framework tool to look up 'crewai' and 'langgraph'. "
        "Return a structured comparison with: "
        "a list of framework names, a 50-word summary of differences, "
        "and a one-sentence recommendation for when to use each."
    ),
    expected_output=(
        "A JSON object with keys: frameworks (list of strings), "
        "summary (string ~50 words), recommendation (string one sentence)."
    ),
    agent=framework_expert,
    output_pydantic=FrameworkComparison,
)

structured_crew = Crew(
    agents=[framework_expert],
    tasks=[structured_task],
    process=Process.sequential,
    verbose=False,
)

structured_result = structured_crew.kickoff()

print("Raw output:")
print(structured_result.raw[:300])
print()

if structured_result.pydantic:
    comp = structured_result.pydantic
    print("Parsed Pydantic model:")
    print(f"  Frameworks:      {comp.frameworks}")
    print(f"  Summary:         {comp.summary}")
    print(f"  Recommendation:  {comp.recommendation}")
else:
    print("(No Pydantic model returned — raw output only)")

In [ ]:
# ─── 6-B: Dynamic kickoff with input variables ────────────────────────────────
# Use {variable} placeholders in task descriptions, then pass values at kickoff.
# This lets you reuse the same crew for different topics without redefining tasks.

parameterized_research = Task(
    description=(
        "Research the latest developments in {topic}. "
        "Focus on the period {time_period}. "
        "Identify exactly {num_trends} key trends with one concrete example each."
    ),
    expected_output="A numbered list of {num_trends} key trends with one concrete example each.",
    agent=researcher,
)

param_crew = Crew(
    agents=[researcher],
    tasks=[parameterized_research],
    process=Process.sequential,
    verbose=False,
)

param_result = param_crew.kickoff(
    inputs={
        "topic": "open-source LLMs",
        "time_period": "Q1-Q2 2025",
        "num_trends": "3",
    }
)

print("=== PARAMETERIZED CREW OUTPUT ===")
print(param_result.raw)

---

## Part 7 — Debugging and Evaluation

---

### Common CrewAI Failure Modes

| Failure | Symptom | Root cause | Fix |
|---------|---------|-----------|-----|
| **Agent ignores the tool** | Output contains hallucinated facts | Weak tool docstring or backstory | Mention the tool by name in backstory + task description |
| **Context not passed** | Writer ignores researcher's output | `context=[task]` missing | Add `context=[research_task]` to dependent tasks |
| **Manager misroutes** | Hierarchical picks wrong agent | Agents' roles are too similar | Sharpen role/goal contrast between agents |
| **Verbose noise** | Log floods in production | `verbose=True` left on | Set `verbose=False` on each Agent and Crew |
| **Infinite delegation loop** | Task never completes | Agents delegate back and forth | Set `allow_delegation=False` on leaf agents |
| **Wrong output format** | Expected JSON, got prose | LLM ignores `expected_output` | Use `output_pydantic` to enforce structure |

### Debugging Checklist

1. Set `verbose=True` on the Agent and Crew — see every reasoning step
2. Inspect `result.tasks_output` — confirm every task ran and produced output
3. Test tools directly with `tool.run(input)` before attaching to agents
4. Print `result.token_usage` — excessive tokens often indicate a delegation loop
5. Verify `context=[...]` is set on every dependent task

In [ ]:
# ─── 7-A: Verbose mode — see every reasoning step ────────────────────────────
# Enable verbose=True on both Agent and Crew to inspect the full chain.
# This is the CrewAI equivalent of LangChain's PromptInspector callback.

debug_agent = Agent(
    role="AI Framework Specialist",
    goal="Accurately explain AI agent frameworks using reference data",
    backstory=(
        "You are a principal engineer. Always use the lookup_framework tool "
        "to verify facts — never rely on memory alone."
    ),
    llm=llm,
    tools=[lookup_framework],
    verbose=True,  # <-- print every reasoning step
)

debug_task = Task(
    description="Look up CrewAI using your lookup_framework tool and summarize it in one sentence.",
    expected_output="One sentence summary of CrewAI.",
    agent=debug_agent,
)

debug_crew = Crew(
    agents=[debug_agent],
    tasks=[debug_task],
    process=Process.sequential,
    verbose=True,  # <-- show crew-level orchestration steps too
)

debug_result = debug_crew.kickoff()
print()
print("=== FINAL ANSWER ===")
print(debug_result.raw)

In [ ]:
# ─── 7-B: Experiment — strong vs weak backstory output quality ────────────────
# A weak backstory produces lower-quality output for the same task.
# Both agents run the same task; compare the specificity of their outputs.

quality_task_desc = (
    f"Research {TOPIC} and produce a bullet-point list of 3 trends. "
    "Each trend must include a specific tool or company name as evidence."
)

weak_agent = Agent(
    role="Researcher",
    goal="Research topics",
    backstory="You are a helpful assistant.",
    llm=llm,
    verbose=False,
)

strong_agent = Agent(
    role="Technology Trend Researcher",
    goal="Find AI and software engineering developments with named evidence",
    backstory=(
        "You are a senior technology analyst with 10 years in AI research. "
        "You always cite specific tools, companies, or research papers as evidence. "
        "Vague claims without examples are unacceptable in your work."
    ),
    llm=llm,
    verbose=False,
)

for label, agent in [("WEAK backstory", weak_agent), ("STRONG backstory", strong_agent)]:
    task = Task(
        description=quality_task_desc,
        expected_output="3 bullet points, each with one named tool or company as evidence.",
        agent=agent,
    )
    crew = Crew(agents=[agent], tasks=[task], process=Process.sequential, verbose=False)
    res = crew.kickoff()
    print(f"=== {label} ===")
    print(res.raw[:400])
    print()

---

## Exercises

---

### Exercise 1 — Add a Fact-Checker Agent

Add a third agent (`fact_checker`) to the research/writing crew. The fact-checker reviews the writer's draft and flags any factual claims that need a source citation. Add a third task with `context=[write_task]`.

**Goal:** Build a 3-task sequential crew where output flows: `researcher → writer → fact_checker`.

**Success criteria:** The fact-checker output should list at least one specific claim (e.g., a statistic or product name) and mark it `NEEDS SOURCE` or `COMMON KNOWLEDGE`.

### Exercise 2 — Convert to Hierarchical

Take the 3-agent crew from Exercise 1 and switch it to `Process.hierarchical`. Compare `result.token_usage` with the sequential version. Does the manager assign tasks in the same order?

### Exercise 3 — Parameterized Crew

Build a crew that accepts any topic and audience at runtime using `kickoff(inputs={...})`. The crew should research the topic and write a summary calibrated to the audience. Test with at least two different `(topic, audience)` pairs.

**Starter template below — fill in the TODO items.**

In [ ]:
# ── Exercise 1 Starter — Add a fact-checker ───────────────────────────────────

# TODO: define fact_checker Agent
# Hint: the backstory should say something like
# "You read technical writing and flag every quantitative claim that needs a citation."
fact_checker = Agent(
    role="TODO: ...",
    goal="TODO: ...",
    backstory="TODO: ...",
    llm=llm,
    verbose=False,
)

# TODO: define fact_check_task
# - description: review the writer's draft for unsupported factual claims
# - expected_output: a numbered list of claims marked NEEDS SOURCE or COMMON KNOWLEDGE
# - context: [write_task]   <-- this is the key line
fact_check_task = Task(
    description="TODO: ...",
    expected_output="TODO: ...",
    agent=fact_checker,
    context=[write_task],
)

# TODO: build the 3-agent crew and call kickoff()
# three_agent_crew = Crew(
#     agents=[researcher, writer, fact_checker],
#     tasks=[research_task, write_task, fact_check_task],
#     process=Process.sequential,
#     verbose=False,
# )
# result_3 = three_agent_crew.kickoff()
# print(result_3.raw)

print("Exercise 1 starter loaded — fill in the TODO items and uncomment the crew.")


# ── Exercise 3 Starter — Parameterized crew ───────────────────────────────────

# TODO: add {audience} to the task descriptions and build the crew
param_research_task_ex = Task(
    description="Research the latest developments in {topic} and summarize for {audience}.",
    expected_output="3 bullet points about {topic} tailored to {audience}.",
    agent=researcher,
)

# TODO: build the crew and run with kickoff(inputs={...})
# param_crew_ex = Crew(...)
# result_ex = param_crew_ex.kickoff(inputs={"topic": "quantum computing", "audience": "business executives"})

print("Exercise 3 starter loaded — fill in the crew and kickoff call.")

---

## What's Next?

You now have a solid foundation in CrewAI. Here's where to go deeper:

### Compare with other frameworks
- **Example 15 — CrewAI Research Crew** ([`../15-crewai-research-crew/`](../15-crewai-research-crew/)): a complete end-to-end research pipeline with live web search tools — the production pattern built on top of what you learned here.
- **Example 19 — Multi-Agent Notebook** ([`../19-multi-agent-notebook/`](../19-multi-agent-notebook/)): the LangGraph supervisor/worker pattern — compare the explicit graph wiring to CrewAI's declarative approach. Understanding both gives you the right tool for each job.
- **Example 21 — AutoGen Debate** ([`../21-autogen-debate/`](../21-autogen-debate/)): a third framework model — direct agent-to-agent conversation without a graph or crew.

### Go deeper in CrewAI
- **Memory**: `Crew(..., memory=True)` enables short-term, long-term, and entity memory across runs
- **Planning**: `Crew(..., planning=True)` adds a planning agent that builds a work plan before executing
- **Async execution**: `crew.kickoff_async()` for non-blocking pipelines in web servers
- **Flows**: `crewai.flow.Flow` for composing multiple crews with conditional branching between them

### Real tool integrations
- `from langchain_community.tools import DuckDuckGoSearchRun` — live web search, no API key needed
- `from crewai_tools import FileReadTool, PDFSearchTool` — file and document tools
- `from crewai_tools import SerperDevTool` — Google search via Serper API

### Further reading
- Hong et al. (2023). *MetaGPT: Meta Programming for a Multi-Agent Collaborative Framework.* https://arxiv.org/abs/2308.00352
- Wu et al. (2023). *AutoGen: Enabling Next-Gen LLM Applications via Multi-Agent Conversation.* https://arxiv.org/abs/2308.08155
- CrewAI docs — Agents: https://docs.crewai.com/concepts/agents
- CrewAI docs — Tasks: https://docs.crewai.com/concepts/tasks
- CrewAI docs — Flows: https://docs.crewai.com/concepts/flows

---

*Workshop complete. The next step is example 15 — build a production research crew with live web tools.*

---

## Exercise Answer Key

Use this section as a reference **after** attempting the exercises yourself. These are sample solutions, not the only correct answers.

---

### Exercise 1 — Add a Fact-Checker Agent

**Key insight:** The `context=[write_task]` line is what connects the fact-checker to the writer's output. Without it, the fact-checker gets no input and produces generic output unrelated to the draft.

**What good output looks like:**
- `1. 'GitHub Copilot has 2 million users' — NEEDS SOURCE`
- `2. 'LLM agents can replace 40% of coding tasks' — NEEDS SOURCE`
- `3. 'Cursor was founded in 2022' — COMMON KNOWLEDGE (approximate)`

If the fact-checker just rephrases the draft, tighten the task description: *"List only claims that are quantitative or specific enough to be verifiably wrong."*

### Exercise 2 — Convert to Hierarchical

**Expected observation:** The manager usually assigns tasks in the same order as sequential for this use case — it's obvious from the task descriptions that research comes before writing. Token usage increases 20-40% due to the extra manager call. Hierarchical mode shows its value when the task-to-agent mapping is genuinely ambiguous.

### Exercise 3 — Parameterized Crew

**Key insight:** Every `{variable}` in `description` and `expected_output` gets replaced by `kickoff(inputs={...})` values. The inputs dict must contain every variable referenced in any task — missing one raises a `KeyError`.

In [ ]:
# ── Exercise 1 Answer — 3-agent sequential crew ───────────────────────────────

fact_checker_answer = Agent(
    role="Scientific Fact Checker",
    goal="Identify factual claims in technical writing that require source citations",
    backstory=(
        "You are a meticulous editor with a background in research methodology. "
        "You read technical writing and flag every claim that is quantitative, "
        "specific, or potentially contestable — anything that needs a citation. "
        "You never rewrite the content; you only flag problems."
    ),
    llm=llm,
    verbose=False,
)

fact_check_task_answer = Task(
    description=(
        "Review the blog introduction from the writer. "
        "List every specific factual claim (statistics, dates, product names, "
        "market share figures) that would need a citation in a published article. "
        "For each claim, mark it NEEDS SOURCE or COMMON KNOWLEDGE."
    ),
    expected_output=(
        "A numbered list of factual claims found in the writing, each marked "
        "NEEDS SOURCE or COMMON KNOWLEDGE. If no specific claims found, "
        "write 'No specific claims requiring citations found.'"
    ),
    agent=fact_checker_answer,
    context=[write_task],  # receives the writer's output
)

three_agent_crew = Crew(
    agents=[researcher, writer, fact_checker_answer],
    tasks=[research_task, write_task, fact_check_task_answer],
    process=Process.sequential,
    verbose=False,
)

result_3 = three_agent_crew.kickoff()

print("=== 3-AGENT CREW: FACT-CHECKER OUTPUT ===")
print(result_3.raw)
print()
print(f"Tasks run: {len(result_3.tasks_output)}")
for i, t in enumerate(result_3.tasks_output):
    print(f"  Task {i + 1} ({t.agent}): {len(t.raw)} chars")

In [ ]:
# ── Exercise 2 Answer — 3-agent hierarchical crew ─────────────────────────────

three_agent_hierarchical = Crew(
    agents=[researcher, writer, fact_checker_answer],
    tasks=[research_task, write_task, fact_check_task_answer],
    process=Process.hierarchical,
    manager_llm=llm,
    verbose=False,
)

result_3h = three_agent_hierarchical.kickoff()

print("=== SEQUENTIAL vs HIERARCHICAL TOKEN USAGE (3-agent) ===")
print(f"Sequential   token usage: {result_3.token_usage}")
print(f"Hierarchical token usage: {result_3h.token_usage}")
print()
print("Hierarchical output (first 300 chars):")
print(result_3h.raw[:300])

In [ ]:
# ── Exercise 3 Answer — Parameterized crew with topic + audience ──────────────

param_research_answer = Task(
    description=(
        "Research the latest developments in {topic}. "
        "Your audience is {audience} — calibrate technical depth accordingly. "
        "Identify 3 key trends with one concrete example each."
    ),
    expected_output=(
        "3 bullet points about {topic} tailored to {audience}. "
        "Each bullet includes one named example or tool."
    ),
    agent=researcher,
)

param_write_answer = Task(
    description=(
        "Write a 100-word summary of {topic} for {audience} "
        "using the research provided. Use language appropriate for the audience — "
        "avoid jargon for non-technical audiences, embrace it for engineers."
    ),
    expected_output="A 100-word summary paragraph about {topic} written for {audience}.",
    agent=writer,
    context=[param_research_answer],
)

param_crew_answer = Crew(
    agents=[researcher, writer],
    tasks=[param_research_answer, param_write_answer],
    process=Process.sequential,
    verbose=False,
)

# Run with two different (topic, audience) combinations
for inputs in [
    {"topic": "quantum computing", "audience": "business executives"},
    {"topic": "Rust programming language", "audience": "junior developers"},
]:
    r = param_crew_answer.kickoff(inputs=inputs)
    print(f"Topic: {inputs['topic']}  |  Audience: {inputs['audience']}")
    print(r.raw[:300])
    print()